# F4a 运行（ clean 版）. 按顺序跑，每格末尾有 EXPECT 行，对不上就停、贴回来。


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
# EXPECT: Mounted at /content/drive


In [ ]:
!rm -rf /content/f4a-bundle && tar xzf /content/drive/MyDrive/f4a-bundle.tar.gz -C /content && ls /content/f4a-bundle
# EXPECT: 看到 train_f4a.py install.sh fetch_data.sh


In [ ]:
from google.colab import userdata
import os
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
open(os.path.expanduser('~/.kaggle/access_token'), 'w').write(userdata.get('ccjjllt').strip())
os.chmod(os.path.expanduser('~/.kaggle/access_token'), 0o600)
for v in ('KAGGLE_USERNAME', 'KAGGLE_KEY'):
    os.environ.pop(v, None)
print('EXPECT: access_token written, legacy env cleared')


In [ ]:
!cd /content/f4a-bundle && bash install.sh
# EXPECT: 末尾 install done. + cuda True Tesla T4


In [ ]:
!cd /content/f4a-bundle && DATA_DIR=./data DRIVE_DIR=/content/drive/MyDrive/f4a-data bash fetch_data.sh
# EXPECT: 末尾 fetch+verify done.（若报 MISSING，先去 Drive 补文件）


In [ ]:
!cd /content/f4a-bundle && python3 train_f4a.py verify --stage cpu --out ./out && python3 train_f4a.py verify --stage data --data . --out ./out
# EXPECT: 两行 ok=True。若 FAIL 就停。


In [ ]:
!cd /content/f4a-bundle && python3 train_f4a.py build-targets --data . --out ./out && ls -lh ./out/targets_cache.npz
# EXPECT: targets ok 114533/114533，且 ls 显示 targets_cache.npz 约 3GB。跑约8-10分钟，o打印 targets ok 后别碰键盘等 ls 出来。


In [ ]:
!cd /content/f4a-bundle && python3 train_f4a.py build-index --data . --out ./out --pool union && ls -lh ./out/formula_index.npz
# EXPECT: formula ok 712199/712199，且 ls 显示 formula_index.npz。


In [ ]:
!cd /content/f4a-bundle && python3 train_f4a.py contrast --data . --out ./out --steps 500
# EXPECT: 末尾 metrics.json 写出，无 Traceback。约15-25分钟，GPU 会动。


In [ ]:
!cd /content/f4a-bundle && tar czf f4a-results.tar.gz out/head_A.pt out/head_B.pt out/metrics.json out/loss_curve_A.json out/loss_curve_B.json out/verify_report.json && ls -lh f4a-results.tar.gz
# EXPECT: f4a-results.tar.gz 若干MB。去 Files 面板下载发回。
